# IPCA – Decomposição Macroeconômica por Blocos

Este notebook utiliza a view **vw_groupsBySubItems** para decompor o IPCA
em blocos macroeconômicos consistentes com a literatura e com o Banco Central:

- Bens industriais
- Serviços
- Serviços subjacentes
- Monitorados
- Alimentação no domicílio
- Alimentação fora do domicílio

Objetivos:
- Construir séries mensais por bloco
- Construir séries acumuladas em 12 meses
- Criar dataset final para modelos estatísticos e ML

Este arquivo é compatível com Jupytext.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.database.connection import engine

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

print("PYTHONPATH atualizado:", ROOT)
pd.set_option("display.float_format", "{:.4f}".format)

# 1. Carregar dados da view `vw_groupsBySubItems`

Esta view já contém:
- grupos
- subgrupos
- itens
- subitens
- contribuições ponderadas
- IPCA cheio

Ela é a base ideal para decomposição macroeconômica.

In [ ]:
query = """
SELECT
    date::date,
    "idGrupos",
    grupos,
    "idSubGrupos",
    "subGrupos",
    "idItens",
    itens,
    "idSubItens",
    "subItens",
    "ipcaSubItensCheio",
    "ipcaSubItensPorPeso"
FROM bcb."vw_groupsBySubItems"
ORDER BY date, "idSubItens";
"""

df = pd.read_sql(query, engine)
df["date"] = pd.to_datetime(df["date"])
df.head()

# 2. Criar mapeamento de blocos macroeconômicos

Aqui definimos os blocos conforme a classificação tradicional do BC.
Ajuste conforme necessário.

In [ ]:
map_blocos = {
    # Bens industriais
    "Eletrodomésticos": "Bens industriais",
    "Vestuário": "Bens industriais",
    "Artigos de residência": "Bens industriais",
    "Transporte público": "Monitorados",

    # Serviços
    "Serviços pessoais": "Serviços",
    "Educação": "Serviços",
    "Saúde": "Serviços",
    "Recreação": "Serviços",

    # Alimentação
    "Alimentação no domicílio": "Alimentação no domicílio",
    "Alimentação fora do domicílio": "Alimentação fora do domicílio",

    # Monitorados
    "Combustíveis": "Monitorados",
    "Energia elétrica": "Monitorados",
    "Água e esgoto": "Monitorados",
}

# fallback
df["bloco_macro"] = df["grupos"].map(map_blocos).fillna("Outros")

df.head()

# 3. Contribuição mensal por bloco

Somamos a contribuição dos subitens dentro de cada bloco.

In [ ]:
df_blocos = (
    df.groupby(["date", "bloco_macro"])["ipcaSubItensPorPeso"]
    .sum()
    .reset_index()
    .pivot(index="date", columns="bloco_macro", values="ipcaSubItensPorPeso")
)

df_blocos.tail()

# 4. Gráfico – Contribuição mensal por bloco

In [ ]:
plt.figure(figsize=(12, 5))
df_blocos.plot(kind="bar", stacked=True, ax=plt.gca(), width=0.9)
plt.title("Contribuição dos Blocos Macroeconômicos para o IPCA (p.p.)")
plt.xlabel("Data")
plt.ylabel("Contribuição (p.p.)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 5. Séries agregadas por bloco (linha)

In [ ]:
plt.figure(figsize=(10, 4))
for col in df_blocos.columns:
    plt.plot(df_blocos.index, df_blocos[col], label=col)

plt.title("Séries Mensais – Contribuição dos Blocos")
plt.xlabel("Data")
plt.ylabel("Contribuição (p.p.)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 6. Contribuição acumulada em 12 meses

Indicador fundamental para análise de tendência.

In [ ]:
df_blocos_12m = df_blocos.rolling(12).sum()
df_blocos_12m.tail()

In [ ]:
plt.figure(figsize=(10, 4))
for col in df_blocos_12m.columns:
    plt.plot(df_blocos_12m.index, df_blocos_12m[col], label=col)

plt.title("Contribuição Acumulada em 12 Meses – Blocos Macroeconômicos")
plt.xlabel("Data")
plt.ylabel("Contribuição acumulada (p.p. em 12m)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 7. Dataset final para modelos estatísticos e ML

Inclui:
- contribuições mensais por bloco
- contribuições acumuladas em 12 meses
- pronto para regressões, VAR, ML etc.

In [ ]:
df_ml = pd.concat(
    [
        df_blocos.add_prefix("bloco_"),
        df_blocos_12m.add_prefix("bloco_12m_"),
    ],
    axis=1,
)

df_ml.tail()

In [ ]:
output_dir = os.path.join(ROOT, "data", "processed")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "ipca_blocos_macro.parquet")
df_ml.to_parquet(output_path)

print("Dataset salvo em:", output_path)

# 8. Conclusões

- A view `vw_groupsBySubItems` permite decompor o IPCA de forma robusta.
- Os blocos macroeconômicos ajudam a entender a **origem da inflação**.
- As séries 12m capturam a **tendência de médio prazo**.
- O dataset final será usado nos notebooks:
  - NB7 (Curva de Phillips)
  - NB8 (Modelos estatísticos)
  - NB9 (Machine Learning)